<a href="https://colab.research.google.com/github/Ed-Neema/Financial-Sentiment-Classification/blob/main/Finance_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install --upgrade transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [ ]:
import torch
print(torch.cuda.is_available())  # should print True

True


In [ ]:
from datasets import load_dataset

# Load the highest-agreement configuration
from datasets import load_dataset

ds = load_dataset("atrost/financial_phrasebank")
print(ds)
print(ds["train"][0])


README.md:   0%|          | 0.00/721 [00:00<?, ?B/s]

data/train-00000-of-00001-138b53eb17a3e8(…): reconstructing file:   0%|          |  0.00B /  268kB            

data/train-00000-of-00001-138b53eb17a3e8(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-0876be41e(…): reconstructing file:   0%|          |  0.00B / 68.7kB            

data/validation-00000-of-00001-0876be41e(…): downloading bytes:           |  0.00B            

...

data/test-00000-of-00001-41c7ea948573445(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/776 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 3100
    })
    validation: Dataset({
        features: ['sentence', 'label'],
        num_rows: 776
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 970
    })
})
{'sentence': 'EBIT margin was up from 1.4 % to 5.1 % .', 'label': 2}


- Must use the exact tokenizer that matches your model.

- DistilBERT learned English using one specific vocabulary — a fixed dictionary where every known word-piece has an assigned ID number. If you fed it text numbered by a different scheme, the numbers would mean nothing to it, like handing someone a book written in an alphabet they've never seen.

- from_pretrained("distilbert/distilbert-base-uncased") downloads that model's own matching vocabulary so the numbers line up with what it learned.

In [ ]:
"""
"""
from transformers import AutoTokenizer
#AutoTokenizer is a convenient class that automatically selects the correct tokenizer for a given pre-trained model.

"""
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased"):
This initializes a tokenizer specifically for the distilbert-base-uncased model. from_pretrained() downloads the pre-trained tokenizer's vocabulary and configuration, ensuring that the tokenization process is consistent with how the model was trained.
'Uncased' means it will convert all text to lowercase.
"""
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

#This defines a function named preprocess_function.
#It takes a dictionary of examples (typically a batch from the dataset) and applies the tokenizer to the "sentence" key within those examples. truncation=True ensures that if a sentence is longer than the tokenizer's maximum input length, it will be cut off to fit, preventing errors.
#each item looks like: {'sentence': 'EBIT margin was up from 1.4 % to 5.1 % .', 'label': 2}
def preprocess_function(examples):
    return tokenizer(examples["sentence"], truncation=True)
tokenized = ds.map(preprocess_function, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/3100 [00:00<?, ? examples/s]

Map:   0%|          | 0/776 [00:00<?, ? examples/s]

Map:   0%|          | 0/970 [00:00<?, ? examples/s]

In [ ]:
"""
The collator needs the tokenizer for two reasons:
- to know which token ID means "padding" (each model reserves a specific number for filler),
- to know which side to pad on. So DataCollatorWithPadding(tokenizer=tokenizer) means "make me a padding machine set up for DistilBERT's rules."
"""
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {"negative": 0, "neutral": 1, "positive": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


- import numpy as np: This line imports the numpy library, which is a fundamental package for numerical computation in Python. It's often used for array manipulations and mathematical operations, which will be useful here for processing predictions.
- import evaluate: This imports the evaluate library, which provides a simple way to load and compute various evaluation metrics commonly used in machine learning.
- accuracy = evaluate.load("accuracy"): This line loads the accuracy metric from the evaluate library. Once loaded, accuracy becomes an object that can be called later to compute the accuracy score.
- def compute_metrics(eval_pred):: This defines a function called compute_metrics. This function is typically passed to a Trainer in the transformers library and is responsible for calculating metrics during evaluation.
- predictions, labels = eval_pred: The eval_pred argument, when passed from a Trainer, is a tuple containing the raw model predictions (logits) and the true labels. This line unpacks those two components into separate variables.
- predictions = np.argmax(predictions, axis=1): Model predictions (logits) are usually raw scores for each class. To get the actual predicted class, you need to find the index of the highest score for each example. np.argmax(predictions, axis=1) does exactly that, converting the raw predictions into a single predicted class ID for each sample.
- return accuracy.compute(predictions=predictions, references=labels): Finally, this line computes the accuracy. It calls the compute method of the accuracy object (which we loaded earlier), passing in the processed predictions (the predicted class IDs) and the labels (the true class IDs or references). The function then returns the calculated accuracy score.

In [ ]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True,
    hub_model_id="Ed-Akoth/financial-sentiment-distilbert",

)

In [ ]:
from huggingface_hub import login
login()  # paste an access token when prompted (from huggingface.co/settings/tokens)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.408110,0.837629
2,No log,0.409200,0.846649
3,0.390930,0.448764,0.846649
4,0.390930,0.564243,0.847938
5,0.390930,0.595608,0.846649


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=970, training_loss=0.2557056702289385, metrics={'train_runtime': 235.5461, 'train_samples_per_second': 65.805, 'train_steps_per_second': 4.118, 'total_flos': 238732242685416.0, 'train_loss': 0.2557056702289385, 'epoch': 5.0})

In [ ]:
test_results = trainer.evaluate(tokenized["test"])
print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy
0.390930,0.431080,5,0.828866


{'eval_loss': 0.43107980489730835, 'eval_accuracy': 0.8288659793814434}
